# Лабораторная работа №4
## Линейные модели, метод опорных векторов и деревья решений

**Тема:** Логистическая регрессия, SVM и дерево решений; оценка качества и интерпретация дерева

**Цель работы:** Сравнить качество линейной (логистической) модели, SVM и дерева решений на одном датасете; построить важность признаков и визуализацию дерева

**Выполнил:** [ФИО студента]  
**Группа:** [Номер группы]  
**Дата:** [Дата выполнения]

## 1. Описание задания

### Задачи лабораторной работы
1. Выбрать набор данных для задачи классификации или регрессии.
2. При необходимости удалить или заполнить пропуски и закодировать категориальные признаки.
3. Разделить выборку на обучающую и тестовую с помощью `train_test_split`.
4. Обучить модели: линейную (для классификации — логистическую регрессию), SVM и дерево решений.
5. Оценить качество двумя подходящими метриками и сравнить модели.
6. Построить график важности признаков для дерева решений.
7. Визуализировать дерево решений или вывести его правила текстом.

В работе используется датасет **Titanic** (бинарная классификация: выжил / не выжил), как и в предыдущих лабораторных — для единообразия предобработки.

## 2. Импорт библиотек

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.svm import SVC
from sklearn.tree import DecisionTreeClassifier, plot_tree, export_text
from sklearn.metrics import accuracy_score, f1_score, classification_report, confusion_matrix

from pathlib import Path
import warnings

warnings.filterwarnings("ignore")

plt.style.use("seaborn-v0_8-darkgrid")
plt.rcParams["figure.figsize"] = (10, 6)
plt.rcParams["font.size"] = 10

print("Библиотеки успешно импортированы")

## 3. Загрузка и предобработка данных

Загружаем данные (при наличии — через `kagglehub`, иначе — по прямой ссылке). Обрабатываем пропуски и категориальные признаки.

In [ ]:
try:
    import kagglehub

    path = kagglehub.dataset_download("c/titanic")
    dataset_path = Path(path)
    csv_files = list(dataset_path.glob("*.csv"))
    train_file = next((f for f in csv_files if "train" in f.name.lower()), csv_files[0])
    df = pd.read_csv(train_file)
    print(f"Датасет загружен: {train_file.name}")
except Exception as e:
    print(f"Загрузка через kagglehub: {e}")
    url = "https://raw.githubusercontent.com/datasciencedojo/datasets/master/titanic.csv"
    df = pd.read_csv(url)
    print("Датасет загружен через альтернативный источник")

print(f"\nРазмерность: {df.shape}")
df.head(10)

In [ ]:
df_clean = df.drop(columns=["PassengerId", "Name", "Ticket", "Cabin"], errors="ignore")

numeric_cols = df_clean.select_dtypes(include=[np.number]).columns.tolist()
categorical_cols = df_clean.select_dtypes(include=["object"]).columns.tolist()

imputer_num = SimpleImputer(strategy="median")
df_clean[numeric_cols] = imputer_num.fit_transform(df_clean[numeric_cols])

for col in categorical_cols:
    mode_vals = df_clean[col].mode()
    fill_val = mode_vals.iloc[0] if len(mode_vals) > 0 else "Unknown"
    df_clean[col] = df_clean[col].fillna(fill_val)

df_encoded = pd.get_dummies(df_clean, columns=categorical_cols, drop_first=True)

X = df_encoded.drop(columns=["Survived"])
y = df_encoded["Survived"]

feature_names = X.columns.tolist()

scaler = StandardScaler()
X_scaled = pd.DataFrame(scaler.fit_transform(X), columns=feature_names, index=X.index)

print("Пропуски после обработки:", X_scaled.isnull().sum().sum())
print(f"Признаков: {X_scaled.shape[1]}, объектов: {X_scaled.shape[0]}")
X_scaled.head()

## 4. Разделение на обучающую и тестовую выборки

Используем `train_test_split` со стратификацией по целевому признаку.

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X_scaled, y, test_size=0.2, random_state=42, stratify=y
)

print(f"Обучающая выборка: {X_train.shape[0]} объектов")
print(f"Тестовая выборка: {X_test.shape[0]} объектов")
print(f"Классы в train: {dict(y_train.value_counts())}")
print(f"Классы в test: {dict(y_test.value_counts())}")

## 5. Обучение моделей

- **Логистическая регрессия** — линейная модель для классификации.
- **SVC** — метод опорных векторов (ядро по умолчанию — RBF); признаки масштабированы.
- **DecisionTreeClassifier** — дерево решений; для интерпретации ограничим `max_depth`, чтобы дерево было компактнее на графике (качество может немного отличаться от «полного» дерева).

In [ ]:
RANDOM_STATE = 42
TREE_MAX_DEPTH = 4  # для читаемой визуализации; увеличьте при необходимости

models = {
    "Логистическая регрессия": LogisticRegression(max_iter=2000, random_state=RANDOM_STATE),
    "SVM (RBF)": SVC(kernel="rbf", random_state=RANDOM_STATE),
    "Дерево решений": DecisionTreeClassifier(
        max_depth=TREE_MAX_DEPTH, random_state=RANDOM_STATE
    ),
}

predictions = {}
fitted = {}

for name, model in models.items():
    model.fit(X_train, y_train)
    predictions[name] = model.predict(X_test)
    fitted[name] = model
    print(f"Обучено: {name}")

## 6. Метрики качества и сравнение моделей

Для задачи бинарной классификации используем:
- **Accuracy** — доля верных предсказаний;
- **F1-score** (взвешенный `average='weighted'`) — баланс точности и полноты с учётом дисбаланса классов.

Дополнительно выводим `classification_report` для дерева (наглядно по классам).

In [ ]:
rows = []
for name in models:
    y_pred = predictions[name]
    rows.append(
        {
            "Модель": name,
            "Accuracy": accuracy_score(y_test, y_pred),
            "F1 (weighted)": f1_score(y_test, y_pred, average="weighted", zero_division=0),
        }
    )

metrics_df = pd.DataFrame(rows).sort_values("F1 (weighted)", ascending=False).reset_index(drop=True)
print("Сводная таблица метрик на тестовой выборке")
print("=" * 55)
print(metrics_df.to_string(index=False))

best = metrics_df.iloc[0]["Модель"]
print(f"\nНаибольшее F1 (weighted): {best}")

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 5))

x = np.arange(len(metrics_df))
axes[0].bar(x, metrics_df["Accuracy"], color="steelblue", alpha=0.85)
axes[0].set_xticks(x)
axes[0].set_xticklabels(metrics_df["Модель"], rotation=15, ha="right")
axes[0].set_ylabel("Accuracy")
axes[0].set_title("Доля верных ответов")
axes[0].set_ylim(0, 1.05)
axes[0].grid(True, alpha=0.3, axis="y")

axes[1].bar(x, metrics_df["F1 (weighted)"], color="coral", alpha=0.85)
axes[1].set_xticks(x)
axes[1].set_xticklabels(metrics_df["Модель"], rotation=15, ha="right")
axes[1].set_ylabel("F1-score")
axes[1].set_title("F1-score (weighted)")
axes[1].set_ylim(0, 1.05)
axes[1].grid(True, alpha=0.3, axis="y")

plt.tight_layout()
plt.show()

In [ ]:
tree_model = fitted["Дерево решений"]
y_pred_tree = predictions["Дерево решений"]

print("Отчёт по классам (дерево решений)")
print(classification_report(y_test, y_pred_tree, target_names=["Не выжил", "Выжил"]))

fig, ax = plt.subplots(figsize=(6, 5))
sns.heatmap(
    confusion_matrix(y_test, y_pred_tree),
    annot=True,
    fmt="d",
    cmap="Blues",
    xticklabels=["Не выжил", "Выжил"],
    yticklabels=["Не выжил", "Выжил"],
    ax=ax,
)
ax.set_xlabel("Предсказание")
ax.set_ylabel("Факт")
ax.set_title("Матрица ошибок — дерево решений")
plt.tight_layout()
plt.show()

## 7. Важность признаков в дереве решений

Атрибут `feature_importances_` суммируется в 1 и отражает вклад признаков в снижение неопределённости (impurity) при разбиениях.

In [ ]:
importances = pd.Series(tree_model.feature_importances_, index=feature_names)
importances = importances.sort_values(ascending=True)

fig, ax = plt.subplots(figsize=(10, max(5, len(importances) * 0.25)))
importances.plot(kind="barh", ax=ax, color="teal", alpha=0.85)
ax.set_xlabel("Важность признака")
ax.set_title("Важность признаков — дерево решений")
plt.tight_layout()
plt.show()

## 8. Визуализация дерева и текстовые правила

Графическое представление (`plot_tree`) и текстовый вывод (`export_text`). При увеличении `max_depth` текст может быть длинным.

In [ ]:
fig, ax = plt.subplots(figsize=(22, 12))
plot_tree(
    tree_model,
    feature_names=feature_names,
    class_names=["Не выжил", "Выжил"],
    filled=True,
    rounded=True,
    fontsize=9,
    ax=ax,
)
ax.set_title("Дерево решений (обучено на масштабированных признаках)")
plt.tight_layout()
plt.show()

In [ ]:
rules = export_text(tree_model, feature_names=list(feature_names))
print(rules)

## 9. Выводы

1. Датасет Titanic подготовлен: заполнены пропуски, категории закодированы one-hot, признаки стандартизированы (важно для логистической регрессии и SVM).
2. Выборка разделена на обучающую и тестовую с `train_test_split` и стратификацией.
3. Обучены три модели; качество оценено метриками **Accuracy** и **F1-score**.
4. Сравнение по таблице и столбчатым диаграммам показывает относительную эффективность методов на выбранной предобработке.
5. Для дерева решений построены график важности признаков, визуализация дерева и текстовые правила.

*Замечание:* дерево обучено на **масштабированных** признаках; пороги в узлах интерпретируются в масштабе после `StandardScaler`. При необходимости интерпретации в исходных единицах обучите отдельное дерево на немасштабированном `X`.